## Generate single source positions per catalogues computing the weighted positions from the multiple detections 
Input: Files generated by 0-extract_data scripts, SUSS6_ra_dec.fits and uvot_ra_dec.fits

Output: SUSS6_ra_dec_per_src.fits and uvot_ra_dec_per_src.fits

In [1]:
from astropy.table import Table
from astropy.coordinates import Longitude
from astropy import units as u
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
roots_tfm = '/home/julien/Documents/Etudes/Astrofisica/Master/TFM/Data'

In [2]:
filenames = ['SUSS6_ra_dec.fits','uvot_ra_dec.fits'] # Name of the input files
projects = ['XMM-Newton','Swift-UVOT'] # Name of the input and output repositories

In [3]:
def create_source_ra_dec_file(filepath):
    """
    Create the single source position catalogue
    
    Note that the three value is hard-coded and corresponds to the 
    minimum subparts that has been necessary to have enough RAM  
    to run the match algorithm.

    :filepath: path to the position catalogue (with multiple detection by source)
    """
    dat = Table.read(filepath)
    df = dat.to_pandas()

    # Treatment of points on the right ascension edge
    # If the standard deviation of the source detections right ascension
    # is higher than 10 deg, the right ascension from the detections near 
    # the maximum boundary (360°) are wrapped near 0° (and get negatives).
    df_source_test = df.groupby('SRCNUM').agg({'RA' : 'std'})
    ind_src_ra_err = df_source_test[df_source_test['RA']>10].index
    if len(ind_src_ra_err)>0:
        ind_ra_err = (df['SRCNUM'].isin(ind_src_ra_err.values)) & (df['RA']>180)
        df.loc[ind_ra_err,'RA']  = df.loc[ind_ra_err,'RA'] - 360   

    # Computation of the weighted column and creation of the utils column to aggregate
    df['RA_weighted'] = df['RA']/df['POSERR']
    df['DEC_weighted'] = df['DEC']/df['POSERR']
    df['1/POSERR'] = 1/df['POSERR'].copy()
    df['COUNT'] = df['SRCNUM'].copy()

    # Aggregation
    df_source = df.groupby('SRCNUM').agg({'RA_weighted' : 'sum', 'DEC_weighted' : 'sum', 
                                         '1/POSERR': 'sum', 'POSERR': 'mean', 'COUNT': 'count'})
    
    # Computation of the weighted mean    
    df_source['DEC_weighted'] = df_source['DEC_weighted']/df_source['1/POSERR']
    df_source.loc[:, 'RA_weighted'] = df_source['RA_weighted']/df_source['1/POSERR']

    # Sanity check
    test = 0
    if df_source.loc[:, 'RA_weighted'].min() < 0:
        test = -1
        print('Sanity check start: Some points have a negative right ascension before correction')
    
    # Points with negative right ascension are moved near the maximum boundary
    df_source.loc[df_source.loc[:, 'RA_weighted']<0, 'RA_weighted'] = df_source.loc[df_source.loc[:, 'RA_weighted']<0, 'RA_weighted'] + 360

    # Sanity check end
    if test == -1:
        if df_source.loc[:, 'RA_weighted'].min() > 0:
            print('Sanity check ok: No negative right ascension remaining after correction')
        else:
            print('Error: Negative right ascension remaining!!')
    
    # Final cleaning
    df_source = df_source.rename(columns={'RA_weighted':'RA','DEC_weighted': 'DEC'}).drop('1/POSERR',axis=1)
   
    # File saving
    t1 = Table.from_pandas(df_source,index=True)
    t1.write(filepath.split(".")[0] +'_per_src.fits',overwrite='True')

In [4]:
for filename,project in zip(filenames, projects):
    filepath = roots_tfm + '/'+project+'/'+filename
    create_source_ra_dec_file(filepath)

Sanity check start: Some points have a negative right ascension before correction
Sanity check ok: No negative right ascension remaining after correction


In [ ]:
# Run test to visualy check the weighted positions with respect 
# to the original positions
filepath_src_uvot = roots_tfm + '/Swift-UVOT/uvot_ra_dec_per_src.fits'
df_source = Table.read(filepath_src_uvot).to_pandas()
filepath_uvot = roots_tfm + '/Swift-UVOT/uvot_ra_dec.fits'
df = Table.read(filepath_uvot).to_pandas()
list_test = df_source.loc[df_source['COUNT']>3,'SRCNUM'].head(10) # Only the first 10 UVOT sources with more than 3 detection are drawn.
for i in list_test:
    plt.figure()
    plt.scatter(df.loc[df.loc[:,'SRCNUM']==i,'RA'],df.loc[df.loc[:,'SRCNUM']==i,'DEC'],color='C0',label='Original')
    plt.scatter(df_source.loc[df_source.loc[:,'SRCNUM']==i,'RA'],df_source.loc[df_source.loc[:,'SRCNUM']==i,'DEC'],color='C1',label='Weighted position')
    ax.set_xlabel("RA [deg]")
    ax.set_ylabel("DEC [deg]")
    plt.legend()